# Post process IAD data and save spectra as CSV files

#ASSUMED_ANISOTROPY = 0.9998 for 1% blood # g=0.98 for blood, g=0.9 standard, g=0.7 for CP phantoms, g=1.0 water

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
from pathlib import Path

In [2]:
# code built by adapting Janek Grohl's DIS post processing code

def read_sis_measurement_data(path, assumed_anisotropy):
    data_file = path + "IAD.txt"
    print(data_file)
    if not os.path.exists(data_file):
        raise FileNotFoundError(f"Did not find file in {data_file}. "
                                f"Please name the result file '1.txt' as per convention.")

    dataframe = pd.read_csv(data_file, sep="\t", skiprows=44, header=None)
    result_dict = dict()
    result_dict["wavelength"] = np.asarray(dataframe[0].values)
    result_dict["absorption"] = np.asarray(dataframe[5].values) * 10  # convert to absorption in 1/cm
    result_dict["reduced_scattering"] = np.asarray(dataframe[6].values) * 10   # convert to reduced scattering in 1/cm
    result_dict["scattering"] = np.asarray(dataframe[6].values) * 10 / (1-assumed_anisotropy)  # convert to scattering in 1/cm
    result_dict["success"] = np.asarray(dataframe[8].values) == "# * "
    
    return result_dict


def post_process(BASE_PATH, ASSUMED_ANISOTROPY):
    
    paths = glob.glob(BASE_PATH + "*")
    all_phantoms = list(dict.fromkeys(paths))

    for phantom_path in all_phantoms:
        PHANTOM = glob.glob(phantom_path + "*")
        NAME = Path(phantom_path).stem

        SAMPLES = []
        for _phantom in PHANTOM:
            SAMPLES = SAMPLES + glob.glob(_phantom + "/*/")
    
        print(NAME)
        print("\t", SAMPLES)
    
        measurements = []
        for folder_name in SAMPLES:
            measurements.append(read_sis_measurement_data(folder_name + "/", assumed_anisotropy=ASSUMED_ANISOTROPY))
    
        good_measurements_x = []
        good_measurements_mua = []
        good_measurements_mus = []
        good_measurements_musp = []
        good_measurement_success = []
        num_samples = 0
        for folder_name in SAMPLES:
            _measurement = read_sis_measurement_data(folder_name, assumed_anisotropy=ASSUMED_ANISOTROPY)
            good_measurements_x.append(_measurement["wavelength"])
            mua = _measurement["absorption"]
            mua[_measurement["success"] == False] = None
            good_measurements_mua.append(mua)
            mus = _measurement["scattering"]
            mus[_measurement["success"] == False] = None
            good_measurements_mus.append(mus)
            musp = _measurement["reduced_scattering"]
            musp[_measurement["success"] == False] = None
            good_measurements_musp.append(musp)
            good_measurement_success.append(_measurement["success"])
            num_samples += 1

        good_measurement_success = np.any(np.reshape(np.asarray(good_measurement_success), (num_samples, -1)), axis=0)
        good_measurements_x = np.nanmedian(np.reshape(np.asarray(good_measurements_x), (num_samples, -1)), axis=0)[good_measurement_success]
        mua_mean = np.nanmedian(np.reshape(np.asarray(good_measurements_mua), (num_samples, -1)), axis=0)[good_measurement_success]
        mus_mean = np.nanmedian(np.reshape(np.asarray(good_measurements_mus), (num_samples, -1)), axis=0)[good_measurement_success]
        musp_mean = np.nanmedian(np.reshape(np.asarray(good_measurements_musp), (num_samples, -1)), axis=0)[good_measurement_success]
        mua_std = np.nanstd(np.reshape(np.asarray(good_measurements_mua), (num_samples, -1)), axis=0)[good_measurement_success]
        mus_std = np.nanstd(np.reshape(np.asarray(good_measurements_mus), (num_samples, -1)), axis=0)[good_measurement_success]
        musp_std = np.nanstd(np.reshape(np.asarray(good_measurements_musp), (num_samples, -1)), axis=0)[good_measurement_success]
        wavelengths = measurements[0]["wavelength"]
        MIN_WL = min(measurements[0]["wavelength"])
        MAX_WL = max(measurements[0]["wavelength"])
    
        # y = Ae^Bx
        B_mus_exp, A_mus_exp = np.polyfit(good_measurements_x, np.log(mus_mean), deg=1)
    

        x_range = np.arange(min(good_measurements_x), max(good_measurements_x), 2) # 2nm step size (SWIR spectrometer samples every 1.6nm)
        mus_interp = np.interp(x_range, good_measurements_x, mus_mean)
        mus_std_interp = np.interp(x_range, good_measurements_x, mus_std)
        
        musp_interp = np.interp(x_range, good_measurements_x, musp_mean)
        musp_std_interp = np.interp(x_range, good_measurements_x, musp_std)
        
        mua_interp = np.interp(x_range, good_measurements_x, mua_mean)
        mua_std_interp = np.interp(x_range, good_measurements_x, mua_std)
  
        for _pname in PHANTOM:
            df = pd.DataFrame({"nm": x_range, "mua[cm-1]": mua_interp, "mua_std[cm-1]": mua_std_interp, 
                               "mus[cm-1]": mus_interp, "mus_std[cm-1]": mus_std_interp, "mus_fit[cm-1]": np.exp(A_mus_exp) * np.exp(B_mus_exp * x_range),
                               "musp[cm-1]": musp_interp, "musp_std[cm-1]": musp_std_interp, "assumed_g": ASSUMED_ANISOTROPY})
            df.to_csv(_pname + "/" + NAME + '.csv', index=False)
    return 


In [3]:
# Ran Tao's postprocessing code for purely absorbing chromophores

class IAD_Result:
    def __init__(self, file_path):
        if file_path is not None:
            self.wavelength_IAD, self.M_R, self.M_R_fit, self.M_T, self.M_T_fit, \
                self.mu_a_IAD, self.mu_sp_IAD = np.loadtxt(file_path, usecols=(0, 1, 2, 3, 4, 5, 6), unpack=True)
            self.error_code = np.loadtxt(file_path, comments=None, skiprows=44, dtype='U1', usecols=9)
            self.mu_a_pure = None  # only for pure absorbing sample

    def refine_mu_a_pure_absorbing(self):
        """
        to refine mu_a spectrum for pure absorbing samples, i.e., remove the mu_a points where mu_sp is not ~0
        :return: None
        """
        self.mu_a_pure = self.mu_a_IAD.copy()
        self.mu_a_pure[self.mu_sp_IAD > 0.05] = np.nan

# Three repeats
BASE_PATH_A = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/H2O_Tao/2025_07_13_DI_H2O_1/IAD.txt')
BASE_PATH_B = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/H2O_Tao/2025_07_13_DI_H2O_2/IAD.txt')
BASE_PATH_C = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/H2O_Tao/2025_07_13_DI_H2O_1/IAD.txt')

A = IAD_Result(BASE_PATH_A)
A.refine_mu_a_pure_absorbing()

B = IAD_Result(BASE_PATH_B)
B.refine_mu_a_pure_absorbing()

C = IAD_Result(BASE_PATH_C)
C.refine_mu_a_pure_absorbing()

Ran_H2O_mua = np.mean((A.mu_a_pure, B.mu_a_pure, C.mu_a_pure), axis=0)*10 # x10 to convert 10 cm-1
Ran_H2O_mua_std = np.std((A.mu_a_pure, B.mu_a_pure, C.mu_a_pure), axis=0)*10 # x10 to convert 10 cm-1

df = pd.DataFrame({"nm": A.wavelength_IAD, "mua[cm-1]": Ran_H2O_mua, "mua_std[cm-1]": Ran_H2O_mua_std}) 
df.to_csv('/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/H2O_Tao/H2O_Tao_mua_only.csv', index=False)


In [4]:
# co-polymer-in-oil, assume g=0.7

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.7)


Co-polymer-in-oil_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW/2025_08_22_co-polymer_A1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW/2025_08_22_co-polymer_B1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW/2025_08_22_co-polymer_C2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW/2025_08_22_co-polymer_C3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Co-polymer-in-oil_MJW/2025_08_22_co-polymer_A2/', '/Users/melissa

In [5]:
# Agar (intralipid and india ink), assume g=0.9

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.9)


Agar_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW/2026_04_02_Agar_A1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW/2026_04_02_Agar_B1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW/2026_04_02_Agar_C2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW/2026_04_02_Agar_C3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Agar_MJW/2026_04_02_Agar_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code

In [6]:
# PDMS (TiO2 and india ink), assume g=0.9

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.9)


PDMS_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW/2026_04_02_PDMS_C1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW/2026_04_02_PDMS_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW/2026_04_02_PDMS_B3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW/2026_04_02_PDMS_B2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/PDMS_MJW/2026_04_02_PDMS_A3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code

In [7]:
# Lard, assume g=0.7

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.7)


Lard_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW/2025_07_15_Lard_C1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW/2025_07_15_Lard_A3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW/2025_07_15_Lard_B2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW/2025_07_15_Lard_B3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Lard_MJW/2025_07_15_Lard_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code

In [8]:
# 1% blood, 0.9998

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.9998)


Raw_1%_blood_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW/2025_07_11_blood_1%_oxy_A3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW/2025_07_11_blood_1%_oxy_B2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW/2025_07_11_blood_1%_oxy_B3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW/2025_07_11_blood_1%_oxy_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_1%_blood_MJW/2025_07_11_blood_1%_oxy_C1/', '/Users/melissa/Library/CloudStorag

/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [9]:
# 1% DEOXY blood, g=0.9998

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.9998)


Deoxy_1%_blood_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW/2025_07_11_blood_1%_deoxy_C1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW/2025_07_11_blood_1%_deoxy_B2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW/2025_07_11_blood_1%_deoxy_A3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW/2025_07_11_blood_1%_deoxy_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_1%_blood_MJW/2025_07_11_blood_1%_deoxy_B3/', '/Users/melis

/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [10]:
# OXY whole blood, g=0.98

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW')

post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.98)


Raw_whole_blood_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW/2025_06_26_whole_blood_A2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW/2025_06_26_whole_blood_A3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW/2025_06_30_whole_blood_C1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW/2025_06_30_whole_blood_B3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Raw_whole_blood_MJW/2025_06_30_whole_blood_B2/', '/Users/melissa/Librar

/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [11]:
# DEOXY whole blood, g=0.98

BASE_PATH = (r'/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW')
  
post_process(BASE_PATH, ASSUMED_ANISOTROPY=0.98)


Deoxy_whole_blood_MJW
	 ['/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW/2025_07_11_wholeblood_deoxy_C3/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW/2025_07_11_wholeblood_deoxy_C2/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW/2025_07_11_wholeblood_deoxy_B1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW/2025_07_11_wholeblood_deoxy_A1/', '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/in-house-SIS/Deoxy_whole_blood_MJW/2025_07_11_wholeblo

/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/Applications/anaconda3/envs/simpa_venv/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
